## Belief Propagation

## Purpose

Belief propagation performs probabilistic inference by passing reusable local summaries, called **messages**, through a factor graph.

It uses the same two operations as variable elimination:

$$
\text{factor multiplication}
\qquad\text{and}\qquad
\text{marginalization}.
$$

The difference is organizational:

- Variable elimination performs the operations for a particular query.
- Belief propagation stores intermediate results as messages and reuses them.

For tree-structured factor graphs, sum-product belief propagation computes exact marginal distributions.

## Factor graphs

For

$$
P(A,B,C)=P(A)P(B\mid A)P(C\mid B),
$$

the factor graph is

```text
f_A ─ A ─ f_AB ─ B ─ f_BC ─ C
```

It has two kinds of nodes:

- variable nodes: $$A,B,C$$,
- factor nodes: $$f_A,f_{AB},f_{BC}$$.

Messages only travel between different node types.


## Variable-to-factor message

$$m_{X\rightarrow f}(x)=
\prod_{h\in N(X)\setminus\{f\}}
m_{h\rightarrow X}(x)
$$

A variable multiplies all incoming messages except the one from the destination factor.

If there are no other incoming messages, the result is a vector of ones.

## Belief at a variable

$$
b_X(x)
\propto
\prod_{f\in N(X)}
m_{f\rightarrow X}(x)
$$

The normalized product of all incoming factor messages is the marginal belief of $X$.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Mapping

import numpy as np

## Compact factor representation

Replace this class with your existing `Factor` class if it already supports multiplication and `sum_out`.

In [2]:
@dataclass(frozen=True)
class Factor:
    variables: tuple[str, ...]
    cardinalities: Mapping[str, int]
    values: np.ndarray

    def __post_init__(self) -> None:
        variables = tuple(self.variables)
        values = np.asarray(self.values, dtype=float)
        expected = tuple(self.cardinalities[v] for v in variables)

        if values.shape != expected:
            raise ValueError(
                f"Expected shape {expected}, got {values.shape}."
            )

        object.__setattr__(self, "variables", variables)
        object.__setattr__(self, "values", values)

    def sum_out(self, variable: str) -> "Factor":
        if variable not in self.variables:
            return self

        axis = self.variables.index(variable)
        return Factor(
            variables=tuple(v for v in self.variables if v != variable),
            cardinalities=self.cardinalities,
            values=self.values.sum(axis=axis),
        )

    def multiply(self, other: "Factor") -> "Factor":
        result_variables = self.variables + tuple(
            v for v in other.variables if v not in self.variables
        )

        def aligned_values(factor: "Factor") -> np.ndarray:
            existing = [
                v for v in result_variables if v in factor.variables
            ]
            permutation = [
                factor.variables.index(v) for v in existing
            ]

            aligned = (
                np.transpose(factor.values, permutation)
                if permutation
                else factor.values
            )

            shape = [
                factor.cardinalities[v] if v in factor.variables else 1
                for v in result_variables
            ]
            return aligned.reshape(shape)

        cardinalities = dict(self.cardinalities)
        cardinalities.update(other.cardinalities)

        return Factor(
            variables=result_variables,
            cardinalities=cardinalities,
            values=aligned_values(self) * aligned_values(other),
        )

    def __mul__(self, other: "Factor") -> "Factor":
        return self.multiply(other)


## Factor graph structure

Each named factor is connected to every variable listed in its scope.

In [4]:
@dataclass
class FactorGraph:
    factors: Mapping[str, Factor]

    def __post_init__(self) -> None:
        self.factors = dict(self.factors)
        self.variable_neighbors: dict[str, list[str]] = {}
        self.factor_neighbors: dict[str, list[str]] = {}

        for factor_name, factor in self.factors.items():
            self.factor_neighbors[factor_name] = list(factor.variables)

            for variable in factor.variables:
                self.variable_neighbors.setdefault(variable, []).append(
                    factor_name
                )


## Sum-product implementation

Messages are cached so that repeated belief queries reuse earlier calculations.

In [5]:
class BeliefPropagation:
    def __init__(self, graph: FactorGraph):
        self.graph = graph
        self.v_to_f: dict[tuple[str, str], np.ndarray] = {}
        self.f_to_v: dict[tuple[str, str], np.ndarray] = {}

    def variable_to_factor(
        self,
        variable: str,
        destination_factor: str,
    ) -> np.ndarray:
        key = (variable, destination_factor)
        if key in self.v_to_f:
            return self.v_to_f[key]

        cardinality = next(
            factor.cardinalities[variable]
            for factor in self.graph.factors.values()
            if variable in factor.variables
        )
        message = np.ones(cardinality)

        for factor_name in self.graph.variable_neighbors[variable]:
            if factor_name == destination_factor:
                continue

            message *= self.factor_to_variable(
                factor_name,
                variable,
            )

        self.v_to_f[key] = message
        return message

    def factor_to_variable(
        self,
        factor_name: str,
        destination_variable: str,
    ) -> np.ndarray:
        key = (factor_name, destination_variable)
        if key in self.f_to_v:
            return self.f_to_v[key]

        working = self.graph.factors[factor_name]

        for variable in self.graph.factor_neighbors[factor_name]:
            if variable == destination_variable:
                continue

            incoming = self.variable_to_factor(
                variable,
                factor_name,
            )

            message_factor = Factor(
                variables=(variable,),
                cardinalities=working.cardinalities,
                values=incoming,
            )
            working = working * message_factor

        for variable in tuple(working.variables):
            if variable != destination_variable:
                working = working.sum_out(variable)

        message = working.values
        self.f_to_v[key] = message
        return message

    def belief(self, variable: str) -> np.ndarray:
        neighbors = self.graph.variable_neighbors[variable]
        cardinality = self.graph.factors[
            neighbors[0]
        ].cardinalities[variable]

        values = np.ones(cardinality)

        for factor_name in neighbors:
            values *= self.factor_to_variable(
                factor_name,
                variable,
            )

        return values / values.sum()


# Example: $A\rightarrow B\rightarrow C$

$$
P(A,B,C)=P(A)P(B\mid A)P(C\mid B)
$$

Binary states are encoded as:

- `0`: False
- `1`: True


In [6]:
cardinalities = {"A": 2, "B": 2, "C": 2}

f_A = Factor(
    variables=("A",),
    cardinalities=cardinalities,
    values=np.array([0.6, 0.4]),
)

f_AB = Factor(
    variables=("A", "B"),
    cardinalities=cardinalities,
    values=np.array([
        [0.8, 0.2],
        [0.3, 0.7],
    ]),
)

f_BC = Factor(
    variables=("B", "C"),
    cardinalities=cardinalities,
    values=np.array([
        [0.9, 0.1],
        [0.2, 0.8],
    ]),
)

graph = FactorGraph({
    "f_A": f_A,
    "f_AB": f_AB,
    "f_BC": f_BC,
})

bp = BeliefPropagation(graph)


## Inspect a message

The message

$$m_{f_{AB}\rightarrow B}(b)=
\sum_A f_{AB}(A,b)m_{A\rightarrow f_{AB}}(A)
$$

summarizes everything on the left side of $B$.

In [7]:
bp.factor_to_variable("f_AB", "B")

array([0.6, 0.4])

## Compute all marginal beliefs

In [8]:
print("P(A):", bp.belief("A"))
print("P(B):", bp.belief("B"))
print("P(C):", bp.belief("C"))


P(A): [0.6 0.4]
P(B): [0.6 0.4]
P(C): [0.62 0.38]


# Evidence as an indicator factor

Observing $C=\text{True}$ can be represented by

$$
\phi_e(C)=[0,1].
$$

The zero removes the state inconsistent with the observation.


In [9]:
e_C_true = Factor(
    variables=("C",),
    cardinalities=cardinalities,
    values=np.array([0.0, 1.0]),
)

evidence_graph = FactorGraph({
    "f_A": f_A,
    "f_AB": f_AB,
    "f_BC": f_BC,
    "e_C": e_C_true,
})

bp_evidence = BeliefPropagation(evidence_graph)

print("P(A | C=True):", bp_evidence.belief("A"))
print("P(B | C=True):", bp_evidence.belief("B"))
print("P(C | C=True):", bp_evidence.belief("C"))


P(A | C=True): [0.37894737 0.62105263]
P(B | C=True): [0.15789474 0.84210526]
P(C | C=True): [0. 1.]


# Trees and loops

## Trees

Belief propagation is exact on trees because there is only one path between any two nodes. Information cannot return through another route and be counted twice.

## Graphs with cycles

Applying the same equations repeatedly on a graph with cycles is called **loopy belief propagation**.

It can be useful, but:

- convergence is not guaranteed,
- converged beliefs may be approximate,
- information may circulate repeatedly.


# Summary

### Variable node

```text
multiply incoming messages
except the destination's message
```

### Factor node

```text
multiply the factor and incoming messages
sum out every variable except the receiver
```

### Belief

```text
multiply all incoming factor messages
normalize
```

The central relationship is:

> Belief propagation distributes variable-elimination calculations across the graph and caches them as reusable messages.
